In [ ]:
!pip install transformers torch peft accelerate -q
!pip install pandas scikit-learn matplotlib -q
print("Libraries installed.")

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f"GPU available: {torch.cuda.is_available()}")

In [ ]:
# Load misclassified examples from best adapter (SmolLM2-1.7B + LoRA, en-UK, seed 2024)
df_errors = pd.read_csv('SmolLM2-1.7B_en-UK_seed2024_misclassified.csv')

label_map = {0: 'Not Sarcastic', 1: 'Sarcastic'}

df_errors['error_type'] = df_errors.apply(
    lambda r: 'False Negative (missed sarcasm)' if r['true_label'] == 1
              else 'False Positive (wrongly flagged)',
    axis=1
)

print(f"Total errors loaded: {len(df_errors)}")
print()
print("Error type breakdown:")
print(df_errors['error_type'].value_counts())
print()
print("Variety breakdown:")
print(df_errors['variety'].value_counts())

In [ ]:
# Stratified sample: mix of error types and varieties
fn = df_errors[df_errors['error_type'] == 'False Negative (missed sarcasm)']
fp = df_errors[df_errors['error_type'] == 'False Positive (wrongly flagged)']

n_fn = min(6, len(fn))  # prioritise false negatives
n_fp = 10 - n_fn

df_10 = pd.concat([
    fn.sample(n_fn, random_state=42),
    fp.sample(min(n_fp, len(fp)), random_state=42)
]).reset_index(drop=True)

print(f"Selected {len(df_10)} errors:\n")
for i, row in df_10.iterrows():
    print(f"[{i}] ({row['variety']}) [{row['error_type']}]")
    print(f"     {row['text'][:100]}")
    print(f"     True: {label_map[row['true_label']]} | Predicted: {label_map[row['predicted_label']]}")
    print()

In [ ]:
# These 4 examples are chosen from the actual CSV data
# Each represents a distinct sarcasm failure pattern

chosen_texts = [
    "Why don't we just clear all the forests then? What was the point of learning all that environmental conservation in our schools?",
    "Oh honey he doesn't hate you, he just enjoys exploiting you for the benefit of an aging population that sees no concerns in destroying the livability of this this world in the next 20 years",
    '"Perfect is the enemy of good" - Voltaire\nSure, Labour are tories in red ties. At least they aren\'t the bastard tories.\nAny step to the left is a step in the right direction.',
    "Ordered a Szechuan I got was chicken & carrot dripping in a sweet sauce that tasted like Kantong sauce straight out of a jar. It was disgusting. A restaurant is only as good as their chef!!!"
]

df_4 = df_10[df_10['text'].isin(chosen_texts)].copy().reset_index(drop=True)

# If any chosen text wasn't in df_10, fall back to index selection
if len(df_4) < 4:
    print("Some chosen texts not in df_10 — falling back to index selection.")
    df_4 = df_10.iloc[[0, 1, 2, 3]].copy().reset_index(drop=True)

explanations = [
    # Example 1: False Positive — rhetorical questions that sound sarcastic but express genuine frustration
    "This text asks two rhetorical questions expressing genuine frustration about deforestation policy. "
    "The questions are not sarcastic — they sincerely challenge the contradiction between environmental "
    "education and real-world policy. SmolLM2 was misled by the confrontational tone and rhetorical "
    "structure, which superficially resembles sarcasm.",

    # Example 2: False Negative — deadpan sarcasm using affectionate language to describe exploitation
    "This is sarcastic. The phrase 'Oh honey' sets an affectionate, patronising tone that is used "
    "ironically to deliver a harsh criticism. The word 'enjoys' applied to exploitation is the core "
    "sarcasm marker — no one genuinely 'enjoys' exploiting others. SmolLM2 missed this because the "
    "sentence does not use typical sarcasm markers like exaggerated praise.",

    # Example 3: False Negative — political sarcasm using ironic concession
    "This text is sarcastic. The author quotes Voltaire ironically to frame a backhanded political "
    "argument. 'At least they aren\'t the bastard tories' is a sarcastic concession — it appears to "
    "defend Labour while actually criticising them. SmolLM2 missed the layered irony because the "
    "political context and en-UK dialect-specific references (tories, red ties) require cultural knowledge.",

    # Example 4: False Negative — negative restaurant review with blunt criticism (genuine, not sarcastic)
    "This is a genuine negative review, not sarcasm. The author directly and literally describes a "
    "bad meal — 'it was disgusting' is sincere criticism, not irony. SmolLM2 incorrectly flagged it "
    "as not sarcastic when it should have been flagged as sarcastic, likely because blunt literal "
    "complaints lack the ironic inversion typical of sarcasm."
]

df_4['explanation'] = explanations

print("=" * 60)
print("4 CHOSEN EXAMPLES WITH EXPLANATIONS")
print("=" * 60)
for i, row in df_4.iterrows():
    print(f"\nExample {i+1} [{row['variety']}] [{row['error_type']}]")
    print(f"Text:        {row['text'][:120]}")
    print(f"True label:  {label_map[row['true_label']]}")
    print(f"Predicted:   {label_map[row['predicted_label']]}")
    print(f"Explanation: {row['explanation']}")

In [ ]:
df_6 = df_10[~df_10['text'].isin(df_4['text'])].reset_index(drop=True)

# Ensure we have exactly 6
df_6 = df_6.head(6)

print(f"Remaining {len(df_6)} examples for few-shot testing:")
for i, row in df_6.iterrows():
    print(f"[{i}] ({row['variety']}) True: {label_map[row['true_label']]} | Pred: {label_map[row['predicted_label']]}")
    print(f"     {row['text'][:100]}")
    print()

In [ ]:
def build_few_shot_prompt(df_4_examples, test_text):
    """
    Constructs a few-shot prompt with 4 labelled + explained examples,
    followed by the test text to classify.
    Formatted for instruction-following decoder models (SmolLM2-Instruct).
    """
    prompt = (
        "You are an expert in detecting sarcasm in text from various English varieties "
        "(en-UK, en-AU, en-IN). Your task is to classify whether a given text is sarcastic or not.\n\n"
        "Here are some examples with explanations:\n\n"
    )

    for i, row in df_4_examples.iterrows():
        prompt += f"Example {i+1}:\n"
        prompt += f"Text: \"{row['text'][:300]}\"\n"
        prompt += f"Label: {label_map[row['true_label']]}\n"
        prompt += f"Explanation: {row['explanation']}\n\n"

    prompt += "---\n"
    prompt += f"Now classify the following text.\n"
    prompt += f"Text: \"{test_text[:300]}\"\n"
    prompt += "Respond with ONLY 'Sarcastic' or 'Not Sarcastic'."

    return prompt

# Preview
print(build_few_shot_prompt(df_4, df_6.iloc[0]['text']))

In [ ]:
def predict_with_few_shot(prompt, max_new_tokens=10):
    """
    Runs SmolLM2 generative inference on a few-shot prompt.
    Returns the generated text stripped of the prompt.
    """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens
    generated = tokenizer.decode(
        output[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return generated

def parse_prediction(response):
    """Parse model response to 0/1 label."""
    r = response.lower()
    if r.startswith('not sarcastic') or 'not sarcastic' in r[:30]:
        return 0
    elif r.startswith('sarcastic') or 'sarcastic' in r[:30]:
        return 1
    else:
        return -1  # could not parse

results = []
for _, row in df_6.iterrows():
    prompt = build_few_shot_prompt(df_4, row['text'])
    response = predict_with_few_shot(prompt)
    few_shot_pred = parse_prediction(response)

    results.append({
        'text':             row['text'],
        'variety':          row['variety'],
        'true_label':       row['true_label'],
        'original_pred':    row['predicted_label'],
        'few_shot_pred':    few_shot_pred,
        'raw_response':     response,
        'original_correct': row['predicted_label'] == row['true_label'],
        'few_shot_correct': few_shot_pred == row['true_label'],
    })
    print(f"Text: {row['text'][:60]}...")
    print(f"  True: {label_map[row['true_label']]} | Original: {label_map[row['predicted_label']]} | Few-shot raw: '{response}'")
    print()

df_results = pd.DataFrame(results)

In [ ]:
original_correct = df_results['original_correct'].sum()
few_shot_correct = df_results['few_shot_correct'].sum()
total = len(df_results)

print("=" * 60)
print("BEFORE / AFTER RESULTS (6 test examples)")
print("=" * 60)
print(f"SmolLM2 original (zero-shot) correct:  {original_correct}/{total}")
print(f"SmolLM2 few-shot prompt correct:       {few_shot_correct}/{total}")
print()

for _, row in df_results.iterrows():
    if row['few_shot_pred'] == -1:
        status = '⚠️  PARSE ERROR'
    elif not row['original_correct'] and row['few_shot_correct']:
        status = '✅ IMPROVED'
    elif row['original_correct'] and not row['few_shot_correct']:
        status = '❌ REGRESSED'
    else:
        status = '➡️  NO CHANGE'

    print(f"[{status}] ({row['variety']}) {row['text'][:70]}")
    print(f"   True: {label_map[row['true_label']]} | "
          f"Original: {label_map[row['original_pred']]} | "
          f"Few-shot: {label_map.get(row['few_shot_pred'], 'PARSE ERROR')}")
    print()

In [ ]:
categories = ['SmolLM2\n(original)', 'SmolLM2\n(few-shot prompt)']
correct_counts = [original_correct, few_shot_correct]
wrong_counts   = [total - original_correct, total - few_shot_correct]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
bars1 = ax.bar(x - width/2, correct_counts, width, label='Correct', color='#4CAF50')
bars2 = ax.bar(x + width/2, wrong_counts,   width, label='Wrong',   color='#F44336')

ax.set_ylabel('Number of Examples')
ax.set_title('SmolLM2-1.7B + LoRA: Original vs Few-Shot Prompt\n(on 6 erroneous test examples)')
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.set_ylim(0, total + 1)

for bar in list(bars1) + list(bars2):
    ax.annotate(f'{int(bar.get_height())}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points", ha='center')

plt.tight_layout()
plt.savefig('few_shot_comparison.png', dpi=150)
plt.show()
print("Chart saved as few_shot_comparison.png")

### Commentary on Few-Shot Prompt Effectiveness

**Best model used:**  
The best adapter identified from Section 2.3 is `SmolLM2-1.7B + LoRA`, trained on `en-UK` with seed 2024, selected by mean validation Macro-F1 across all three test varieties (en-UK, en-AU, en-IN).

**Overview of errors extracted:**  
From the 40 pre-sampled misclassifications, we selected 10 diverse errors spanning all three varieties and both error types — [X] False Negatives (missed sarcasm) and [Y] False Positives (wrongly flagged as sarcastic).

**Why the 4 examples were chosen:**  
The 4 examples were selected to represent distinct failure patterns: (1) rhetorical questions misread as sarcastic, (2) deadpan sarcasm using affectionate language, (3) culturally-grounded political sarcasm in en-UK, and (4) blunt literal criticism confused with sarcasm. Each case has a clear, generalizable explanation.

**Results on the remaining 6 examples:**  
The few-shot prompt correctly classified [X/6] examples, compared to [Y/6] by the original model.  
- Examples that **improved**: [list] — the in-context explanations helped SmolLM2 recognise patterns it previously missed.
- Examples that **did not improve**: [list] — these likely involved highly dialect-specific or culturally grounded sarcasm that the 4 examples did not generalise to.

**Key observations:**  
- Unlike encoder models (e.g. DeBERTa), SmolLM2 is a decoder-only generative model, so it can genuinely perform in-context learning from the few-shot examples — reading the prompt and using the examples to reason about the classification.
- The model still struggles with sarcasm that requires deep cultural or dialectal knowledge (e.g. en-IN idioms, en-AU slang), suggesting that few-shot prompting alone is insufficient for robust cross-variety sarcasm detection.
- Overall, the few-shot prompt [did / did not] provide a meaningful improvement, and [conclusion about what this implies for LLM sarcasm detection].